# Phase 2D.1C — MMDetection Dataset Loading Reproducibility

Notebook này tái lập kiểm định Phase 2D.1C từ snapshot mã nguồn đã khóa:

- Commit: `0bf30cb phase2D1C: validate MMDetection dataset loading`
- Phạm vi: 4.894 ảnh, 36.096 annotation, 14 lớp, 500 ảnh zero-GT
- Mục tiêu: kiểm tra tính toàn vẹn đầu vào, unit tests, integration và full pipeline audit
- Trạng thái quyết định: `dataset_training_ready=true`, `training_authorized=false`

Chạy tuần tự từ trên xuống. Không dùng `git pull`, không sửa source và không commit/push trong notebook này.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 1. Khóa repository tại commit chính thức


In [ ]:
%%bash
set -euo pipefail

REPO_URL="https://github.com/kelvinquach/ssl_detection_xray_v2.git"
REPO_DIR="/content/ssl_detection_xray_v2"
LOCKED_COMMIT="0bf30cb"

if [ -d "$REPO_DIR" ]; then
    rm -rf "$REPO_DIR"
fi

git clone "$REPO_URL" "$REPO_DIR"
cd "$REPO_DIR"
git checkout --detach "$LOCKED_COMMIT"

ACTUAL_COMMIT="$(git rev-parse --short=7 HEAD)"
EXPECTED_COMMIT="$(git rev-parse --short=7 "$LOCKED_COMMIT")"

test "$ACTUAL_COMMIT" = "$EXPECTED_COMMIT"
test -z "$(git status --porcelain)"

echo "Repository: $REPO_URL"
echo "Locked commit: $(git log -1 --oneline)"
echo "Working tree: clean"
echo "PHASE 2D.1C COMMIT LOCK: PASS"


In [ ]:
%%bash
set -euo pipefail

cd /content/ssl_detection_xray_v2

EXPECTED_COMMIT="0bf30cb"
ACTUAL_COMMIT="$(git rev-parse --short=7 HEAD)"

test "$ACTUAL_COMMIT" = "$EXPECTED_COMMIT"
test -z "$(git status --porcelain)"

echo "Commit verified: $ACTUAL_COMMIT"
echo "Repository clean: PASS"
echo "REPRODUCIBILITY PRECHECK: PASS"


In [ ]:
from pathlib import Path
import os
import subprocess

REPO = Path("/content/ssl_detection_xray_v2")
LOCKED_COMMIT = "0bf30cb"

assert REPO.is_dir(), f"Không tìm thấy repository: {REPO}"

actual_commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "--short=7", "HEAD"],
    text=True,
).strip()

assert actual_commit == LOCKED_COMMIT, (
    f"Sai commit: expected={LOCKED_COMMIT}, actual={actual_commit}"
)

os.chdir(REPO)

print(f"Working directory: {Path.cwd()}")
print(f"Locked commit: {actual_commit}")
print("NOTEBOOK REPOSITORY SETUP: PASS")


## 2. Cài đặt môi trường MMDetection 3.3.0


In [ ]:
%%bash
set -euo pipefail

cd /content/ssl_detection_xray_v2
test -f scripts/setup_mmdet330_colab.sh

bash scripts/setup_mmdet330_colab.sh /content/ssl_detection_xray_v2

echo "MMDETECTION ENVIRONMENT SETUP: PASS"


## 3. Kiểm tra và nạp gói dữ liệu đầu vào từ Google Drive


In [ ]:
# kiểm tra file có tồn tại trên drive
!ls -lh "/content/drive/MyDrive/ssl_detection_xray_v2/phase2D1C_colab_input.zip"

In [ ]:
# KIỂM TRA TÍNH TOÀN VẸN ZIP TRƯỚC KHI GIẢI NÉN
# DÒNG CUỐI PHẢI CÓ "No errors detected in compressed data"
!unzip -t "/content/drive/MyDrive/ssl_detection_xray_v2/phase2D1C_colab_input.zip" > /tmp/phase2D1C_zip_test.log
!tail -n 2 /tmp/phase2D1C_zip_test.log

In [ ]:
# CHẠY HAI LỆNH ĐỂ GIẢI NÉN TRỰC TIẾP VÀO REPOSITORY
%cd /content/ssl_detection_xray_v2
!unzip -q "/content/drive/MyDrive/ssl_detection_xray_v2/phase2D1C_colab_input.zip" -d /content/ssl_detection_xray_v2

In [ ]:
# KIỂM TRA DỮ LIỆU
# SAU ĐÓ KIỂM TRA CHÍNH XÁC DỮ LIỆU KẾT QUẢ BẮT BUỘC PHẢI CÓ
# 4894
# COCO JSON: FOUND
!find data/processed/images_jpg/train -type f -iname "*.jpg" | wc -l
!test -f data/processed/coco/coco_master_jpg.json && echo "COCO JSON: FOUND"
!du -sh data/processed/images_jpg/train
!sha256sum data/processed/coco/coco_master_jpg.json

## 4. Kiểm tra tính toàn vẹn dữ liệu COCO/JPG


In [ ]:
%%bash
# để kết luận MMDetection dataset hợp lệ. wc -l chỉ chứng minh có 4.894 file .jpg
# không chứng minh JSON có đúng 4.894 ảnh, 36.096 annotation, 14 lớp và mọi file_name đều tồn tại.
### CHẠY LỆNH BÊN DƯỚI BẮT BUỘC PHẢI CÓ
### COCO images: 4894
### COCO annotations: 36096
### COCO categories: 14
### Unique resolved JPG paths: 4894
### Missing referenced JPG files: 0
### Invalid annotation image references: 0
### Invalid annotation category references: 0
### Zero-GT images: 500
## PHASE 2D.1C INPUT INTEGRITY: PASS
set -euo pipefail

REPO=/content/ssl_detection_xray_v2
PY=/content/miniconda/envs/mmdet330/bin/python

cd "$REPO"

"$PY" - <<'PY'
import json
from pathlib import Path

repo = Path("/content/ssl_detection_xray_v2")
ann_path = repo / "data/processed/coco/coco_master_jpg.json"
image_root = repo / "data/processed/images_jpg/train"

with ann_path.open("r", encoding="utf-8") as f:
    coco = json.load(f)

images = coco.get("images", [])
annotations = coco.get("annotations", [])
categories = coco.get("categories", [])

assert len(images) == 4894, len(images)
assert len(annotations) == 36096, len(annotations)
assert len(categories) == 14, len(categories)

image_ids = [x["id"] for x in images]
category_ids = [x["id"] for x in categories]

assert len(image_ids) == len(set(image_ids)), "Duplicate image IDs"
assert len(category_ids) == len(set(category_ids)), "Duplicate category IDs"

image_id_set = set(image_ids)
category_id_set = set(category_ids)

invalid_image_refs = [
    ann["id"] for ann in annotations
    if ann["image_id"] not in image_id_set
]
invalid_category_refs = [
    ann["id"] for ann in annotations
    if ann["category_id"] not in category_id_set
]

missing_files = []
resolved_paths = []

for image in images:
    file_name = Path(image["file_name"])

    # Hỗ trợ cả "train/x.jpg" và chỉ "x.jpg".
    candidates = [
        repo / "data/processed/images_jpg" / file_name,
        image_root / file_name,
        image_root / file_name.name,
    ]

    resolved = next((p for p in candidates if p.is_file()), None)
    if resolved is None:
        missing_files.append(image["file_name"])
    else:
        resolved_paths.append(resolved.resolve())

zero_gt_image_ids = image_id_set - {
    ann["image_id"] for ann in annotations
}

assert not invalid_image_refs, (
    f"Annotations reference invalid image IDs: {len(invalid_image_refs)}"
)
assert not invalid_category_refs, (
    f"Annotations reference invalid category IDs: "
    f"{len(invalid_category_refs)}"
)
assert not missing_files, (
    f"Missing files referenced by COCO JSON: {len(missing_files)}; "
    f"examples={missing_files[:10]}"
)
assert len(set(resolved_paths)) == 4894, (
    "COCO image records do not resolve to 4,894 unique JPG files"
)
assert len(zero_gt_image_ids) == 500, len(zero_gt_image_ids)

print("COCO images:", len(images))
print("COCO annotations:", len(annotations))
print("COCO categories:", len(categories))
print("Unique resolved JPG paths:", len(set(resolved_paths)))
print("Missing referenced JPG files:", len(missing_files))
print("Invalid annotation image references:", len(invalid_image_refs))
print("Invalid annotation category references:", len(invalid_category_refs))
print("Zero-GT images:", len(zero_gt_image_ids))
print("PHASE 2D.1C INPUT INTEGRITY: PASS")
PY

In [ ]:
%%bash
# 1. Kiểm tra file và dữ liệu
cd /content/ssl_detection_xray_v2 || exit 1

status=0

for path in \
  scripts/02D1C_validate_mmdet_dataset_loading.py \
  tests/test_phase2D1C_mmdet_dataset_loading_guardrails.py \
  configs/validation/phase2D1C_mmdet_dataset_loading.py \
  data/processed/coco/coco_master_jpg.json
do
  if test -f "$path"; then
    echo "FOUND FILE: $path"
  else
    echo "MISSING FILE: $path"
    status=1
  fi
done

if test -d data/processed/images_jpg/train; then
  echo "FOUND DIRECTORY: data/processed/images_jpg/train"
else
  echo "MISSING DIRECTORY: data/processed/images_jpg/train"
  status=1
fi

if test "$status" -eq 0; then
  echo "PHASE 2D.1C FILE PREFLIGHT: PASS"
else
  echo "PHASE 2D.1C FILE PREFLIGHT: FAIL"
  exit 1
fi

## 5. Static checks và unit tests


In [ ]:
%%bash
# 2. Static và unit tests
set -e

cd /content/ssl_detection_xray_v2

PYTHON=/content/miniconda/envs/mmdet330/bin/python

"$PYTHON" -m py_compile \
  scripts/02D1C_validate_mmdet_dataset_loading.py

"$PYTHON" -m pytest \
  tests/test_phase2D1C_mmdet_dataset_loading_guardrails.py \
  -q -rs

echo "PHASE 2D.1C STATIC/UNIT VERIFICATION: PASS"

## 6. Integration và full pipeline audit


In [ ]:
%%bash
# 3. Chạy lại integration với MPLBACKEND=Agg
set -e

cd /content/ssl_detection_xray_v2

PYTHON=/content/miniconda/envs/mmdet330/bin/python

MPLBACKEND=Agg "$PYTHON" \
  scripts/02D1C_validate_mmdet_dataset_loading.py \
  --repo-root /content/ssl_detection_xray_v2 \
  --ann-file data/processed/coco/coco_master_jpg.json \
  --data-root data/processed/images_jpg \
  --batch-size 1 \
  --num-workers 0 \
  --seed 42 \
  --expected-images 4894 \
  --expected-annotations 36096 \
  --expected-categories 14 \
  --expected-empty-images 500 \
  --strict

echo "PHASE 2D.1C INTEGRATION PROCESS EXIT: 0"

In [ ]:
%%bash
# 4 CHẠY TOÀN BỘ --full-pipeline-audit ĐỂ chứng minh integration kiểm định toàn bộ 4.894 ảnh
set -e

cd /content/ssl_detection_xray_v2

PYTHON=/content/miniconda/envs/mmdet330/bin/python

MPLBACKEND=Agg "$PYTHON" \
  scripts/02D1C_validate_mmdet_dataset_loading.py \
  --repo-root /content/ssl_detection_xray_v2 \
  --ann-file data/processed/coco/coco_master_jpg.json \
  --data-root data/processed/images_jpg \
  --batch-size 1 \
  --num-workers 0 \
  --seed 42 \
  --expected-images 4894 \
  --expected-annotations 36096 \
  --expected-categories 14 \
  --expected-empty-images 500 \
  --full-pipeline-audit \
  --strict

echo "PHASE 2D.1C FULL PIPELINE AUDIT PROCESS EXIT: 0"

In [ ]:
# KIỂM TRA REPORT

import json
from pathlib import Path
from pprint import pprint

report_path = Path(
    "/content/ssl_detection_xray_v2/"
    "reports/phase2D1C_mmdet_dataset_loading_report.json"
)

report = json.loads(report_path.read_text(encoding="utf-8"))

print("=== FULL PIPELINE AUDIT EVIDENCE ===")
pprint(report.get("pipeline_validation_summary"), sort_dicts=False)
pprint(report.get("bbox_label_validation_summary"), sort_dicts=False)

print("\n=== FINAL STATUS ===")
print("Errors:", len(report.get("errors", [])))
print("dataset_training_ready:", report.get("dataset_training_ready"))
print("training_authorized:", report.get("training_authorized"))

errors = report.get("errors", [])
assert len(errors) == 0, f"Report contains errors: {errors[:5]}"
assert report.get("dataset_training_ready") is True
assert report.get("training_authorized") is False

print("\nPHASE 2D.1C FINAL REPORT CHECK: PASS")


## Tiêu chí hoàn tất

Notebook chỉ đạt PASS khi các cell xác nhận đồng thời:

- commit hiện tại là `0bf30cb`;
- dữ liệu có 4.894 ảnh, 36.096 annotation, 14 lớp và 500 ảnh zero-GT;
- không thiếu JPG hoặc tham chiếu annotation/category;
- unit tests PASS;
- integration và full pipeline audit thoát với mã 0;
- report kết luận `dataset_training_ready=true`, `training_authorized=false`, `Errors: 0`.

`training_authorized=false` là trạng thái đúng của Phase 2D.1C; notebook này không cấp quyền bắt đầu training.
